Agregue esto porque GoogleColab estaba teniendo problems de versiones

In [1]:
!pip uninstall torch -y
!pip uninstall torchvision torchaudio -y
!pip install torch torchvision torchaudio --no-cache-dir

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 149.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 228.0 MB/s eta 0:00:00


In [2]:
import torch

print(torch.__version__)
print(torch.rand(2,2))

2.10.0+cu128
tensor([[0.7244, 0.6774],
        [0.1185, 0.1126]])


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
data = fetch_california_housing(as_frame=True)
df = data.frame

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"].values
feature_names = X.columns

X = X.values

 Split (80-10-10 )

In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1111, random_state=42)

Quitamos los outlier sin tocar latitude y longitude

In [6]:
def fit_outlier_bounds(X, feature_names, exclude_features=['Latitude', 'Longitude']):
    df = pd.DataFrame(X, columns=feature_names)
    bounds = {}

    for col in feature_names:
        if col in exclude_features:
            continue

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        bounds[col] = (lower, upper)

    return bounds


def remove_outliers(X, y, feature_names, bounds):
    df = pd.DataFrame(X, columns=feature_names)
    df['target'] = y

    mask = pd.Series([True] * len(df))

    for col, (lower, upper) in bounds.items():
        mask &= (df[col] >= lower) & (df[col] <= upper)

    df_clean = df[mask]

    return df_clean.drop(columns=['target']).values, df_clean['target'].values

Quitamos los Outliers de los valores entrenados, con los limites obtenidos (funcion de clase)

In [7]:
bounds = fit_outlier_bounds(X_train, feature_names)
X_train, y_train = remove_outliers(X_train, y_train, feature_names, bounds)

Preprocesamiento en PyTorch (nn.Module)

In [8]:
class ScalingLayer(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)

    def forward(self, X):
        return (X - self.mean) / (self.std + 1e-8)

Ahora aplicamos el standard scaling (0-1) para evitar el data leakage

In [9]:
class CustomPreprocessing(nn.Module):
    def __init__(self, mean, std, feature_names):
        super().__init__()

        self.scaler = ScalingLayer(mean, std)

        self.exclude_idx = [
            feature_names.get_loc("Latitude"),
            feature_names.get_loc("Longitude")
        ]

    def forward(self, X):
        X_scaled = self.scaler(X)
        X_scaled[:, self.exclude_idx] = X[:, self.exclude_idx]
        return X_scaled

In [10]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)

mean = X_train_t.mean(dim=0)
std = X_train_t.std(dim=0)

Modelo - Generamos las redes neuronales para cada modelo esto para tener diferente estructura y hiperparametros

In [11]:
class NeuralNetworkM1(nn.Module):
    def __init__(self, input_size, mean, std, feature_names, hidden_size=64):
        super().__init__()

        # Preprocesamiento
        self.scaler = ScalingLayer(mean, std)

        self.exclude_idx = [
            feature_names.get_loc("Latitude"),
            feature_names.get_loc("Longitude")
        ]


        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):

        x_scaled = self.scaler(x)
        x_scaled[:, self.exclude_idx] = x[:, self.exclude_idx]

        return self.net(x_scaled)

In [12]:
class NeuralNetworkM2(nn.Module):
    def __init__(self, input_size, mean, std, feature_names, hidden_size=128):
        super().__init__()

        self.scaler = ScalingLayer(mean, std)

        self.exclude_idx = [
            feature_names.get_loc("Latitude"),
            feature_names.get_loc("Longitude")
        ]

        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Linear(hidden_size//2, 1)
        )

    def forward(self, x):
        x_scaled = self.scaler(x)
        x_scaled[:, self.exclude_idx] = x[:, self.exclude_idx]

        return self.net(x_scaled)

In [13]:
class NeuralNetworkM3(nn.Module):
    def __init__(self, input_size, mean, std, feature_names, hidden_size=128):
        super().__init__()

        self.scaler = ScalingLayer(mean, std)

        self.exclude_idx = [
            feature_names.get_loc("Latitude"),
            feature_names.get_loc("Longitude")
        ]

        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size * 2),
            nn.ReLU(),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        x_scaled = self.scaler(x)
        x_scaled[:, self.exclude_idx] = x[:, self.exclude_idx]

        return self.net(x_scaled)

Entrenamiento - Convertimos a tensores y obtenemos valor de regresión

In [14]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=50, lr=0.001):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1,1)

    for epoch in range(epochs):
        model.train()

        optimizer.zero_grad()
        preds = model(X_train_t)
        loss = criterion(preds, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t)

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Train: {loss.item():.4f} | Val: {val_loss.item():.4f}")

    return model

Evaluación

In [15]:
def evaluate(model, X, y):
    model.eval()

    X_t = torch.tensor(X, dtype=torch.float32)

    with torch.no_grad():
        preds = model(X_t).numpy().flatten()

    mae = mean_absolute_error(y, preds)
    mse = mean_squared_error(y, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y, preds)

    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

Entrenar 3 modelos

In [16]:
print("===== MODELO 1 =====")
model1 = NeuralNetworkM1(X_train.shape[1], mean, std, feature_names)
model1 = train_model(model1, X_train, y_train, X_val, y_val)
val_metrics_1 = evaluate(model1, X_val, y_val)
print("Model 1:", val_metrics_1)

print("===== MODELO 2 =====")
model2 = NeuralNetworkM2(X_train.shape[1], mean, std, feature_names)
model2 = train_model(model2, X_train, y_train, X_val, y_val)
val_metrics_2 = evaluate(model2, X_val, y_val)
print("Model 2:", val_metrics_2)

print("===== MODELO 3 =====")
model3 = NeuralNetworkM3(X_train.shape[1], mean, std, feature_names)
model3 = train_model(model3, X_train, y_train, X_val, y_val)
val_metrics_3 = evaluate(model3, X_val, y_val)
print("Model 3:", val_metrics_3)

===== MODELO 1 =====
Epoch 0 | Train: 207.5764 | Val: 179.9373
Epoch 10 | Train: 11.4683 | Val: 8.9090
Epoch 20 | Train: 9.2631 | Val: 11.9077
Epoch 30 | Train: 4.8989 | Val: 6.3381
Epoch 40 | Train: 1.2004 | Val: 5.4594
Model 1: {'MAE': 1.035104938620338, 'MSE': 6.270322813201756, 'RMSE': np.float64(2.504061263867511), 'R2': -3.89738999581581}
===== MODELO 2 =====
Epoch 0 | Train: 10.8289 | Val: 2.0979
Epoch 10 | Train: 2.5527 | Val: 2.3802
Epoch 20 | Train: 1.1659 | Val: 1.5732
Epoch 30 | Train: 1.3082 | Val: 1.5299
Epoch 40 | Train: 1.1047 | Val: 1.4753
Model 2: {'MAE': 0.847999900664926, 'MSE': 1.5532287698962113, 'RMSE': np.float64(1.2462859904115955), 'R2': -0.2131380258265887}
===== MODELO 3 =====
Epoch 0 | Train: 9.1869 | Val: 1.9388
Epoch 10 | Train: 1.1616 | Val: 1.3294
Epoch 20 | Train: 1.1873 | Val: 1.2554
Epoch 30 | Train: 1.1626 | Val: 1.2580
Epoch 40 | Train: 1.1252 | Val: 1.2294
Model 3: {'MAE': 0.8423829165549795, 'MSE': 1.202070093289601, 'RMSE': np.float64(1.09638957

In [17]:
models = [
    ("Model 1", model1, val_metrics_1),
    ("Model 2", model2, val_metrics_2),
    ("Model 3", model3, val_metrics_3)
]

best_name, best_model, best_metrics = sorted(models, key=lambda x: x[2]["RMSE"])[0]

print("Best model:", best_name)
print("Validation metrics:", best_metrics)

Best model: Model 3
Validation metrics: {'MAE': 0.8423829165549795, 'MSE': 1.202070093289601, 'RMSE': np.float64(1.096389571862849), 'R2': 0.06113190269069402}


Evaluación final

In [18]:
test_metrics = evaluate(best_model, X_test, y_test)

print("Test metrics:", test_metrics)

Test metrics: {'MAE': 0.8614989313583411, 'MSE': 1.2463588283250209, 'RMSE': np.float64(1.1164044196996987), 'R2': 0.06542435263122104}


#Reflexión

Tuve algunos problemas con la implementación, me estaba enredando un poco sobre como debería de ser la estructura, por lo que se puede dar cuenta que eliminé todo el EDA jajjaj. De este modo pude tener una mejor visión sobre las celdas más importantes<br>Al parecer ya utilizaba el standard scaling, porque recuerdo que en deep learning tenía que hacer un proyecto pero el dataset tenía datos muy volátiles y para ello los convertí en un rango de 0-1, pero no sabía que ya existía como una función para ello. <br> Con respecto a los modelos llego a la misma conclusión que la tarea pasada, no siempre el mejor modelo es el que tiene una arquitectura más robusta, sino el que tiene mejor combinación de hiperparámetros.